# Notebook 06 — Learner Error Mining & Diagnostic Difficulty

## Purpose
Extract structured learner-performance evidence from official DBE
NSC Mathematics Diagnostic Reports (2023–2025).

## This notebook produces
1. Diagnostic document register
2. Question-level performance indicators (where available)
3. Common error / misconception records
4. Topic-linked difficulty signals

## Methodological rule
Learner difficulty claims must come from diagnostic evidence,
not from exam frequency alone.

In [6]:
from pathlib import Path
import pandas as pd
import re
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "diagnostic_reports"
META_DIR = PROJECT_ROOT / "data" / "metadata"
PROC_DIR = PROJECT_ROOT / "data" / "processed" / "diagnostics"

for d in [RAW_DIR, META_DIR, PROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Raw diagnostics folder:", RAW_DIR)
print("Exists:", RAW_DIR.exists())
print("Files found:")
for p in sorted(RAW_DIR.glob("*.pdf")):
    print(" -", p.name)

Raw diagnostics folder: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\diagnostic_reports
Exists: True
Files found:
 - 2024 NSC Diagnostics Book 1.pdf
 - 2025 Diagnostic Report - Book 1.pdf
 - Diagnostics Report 2023 Book 1.pdf


In [7]:
register_rows = [
    {
        "document_id": "2023_maths_diagnostic",
        "year": 2023,
        "source": "DBE",
        "report_type": "NSC Diagnostic Report Book 1 - Mathematics",
        "status": "queued",
        "local_path": "",
        "notes": "Mathematics chapter only",
    },
    {
        "document_id": "2024_maths_diagnostic",
        "year": 2024,
        "source": "DBE",
        "report_type": "NSC Diagnostic Report Book 1 - Mathematics",
        "status": "queued",
        "local_path": "",
        "notes": "Mathematics chapter only",
    },
    {
        "document_id": "2025_maths_diagnostic",
        "year": 2025,
        "source": "DBE",
        "report_type": "NSC Diagnostic Report Book 1 - Mathematics",
        "status": "queued",
        "local_path": "",
        "notes": "Mathematics chapter only",
    },
]

diag_register = pd.DataFrame(register_rows)
diag_register_path = META_DIR / "diagnostic_document_register.csv"
diag_register.to_csv(diag_register_path, index=False)
print("Saved:", diag_register_path)
display(diag_register)

Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\diagnostic_document_register.csv


,document_id,year,source,report_type,status,local_path,notes
0,2023_maths_diagnostic,2023,DBE,NSC Diagnostic Report Book 1 - Mathematics,queued,,Mathematics chapter only
1,2024_maths_diagnostic,2024,DBE,NSC Diagnostic Report Book 1 - Mathematics,queued,,Mathematics chapter only
2,2025_maths_diagnostic,2025,DBE,NSC Diagnostic Report Book 1 - Mathematics,queued,,Mathematics chapter only


In [8]:
error_schema = [
    "diagnostic_id",
    "year",
    "paper",          # P1 / P2 / both / unknown
    "question_number",
    "subquestion",
    "topic",          # mapped later if possible
    "avg_performance_pct",  # if reported
    "performance_band",     # poor / average / good / unknown
    "common_error",
    "misconception",
    "suggestion",
    "evidence_span",
    "source_document",
    "extraction_method",
    "confidence",
]

print("Target error schema:")
for c in error_schema:
    print(" -", c)

Target error schema:
 - diagnostic_id
 - year
 - paper
 - question_number
 - subquestion
 - topic
 - avg_performance_pct
 - performance_band
 - common_error
 - misconception
 - suggestion
 - evidence_span
 - source_document
 - extraction_method
 - confidence


In [9]:
from pathlib import Path
RAW_DIR = Path.cwd().parent / "data" / "raw" / "diagnostic_reports"
pdfs = sorted(RAW_DIR.glob("*.pdf"))
print("PDFs ready:", len(pdfs))
for p in pdfs:
    print("-", p.name)

PDFs ready: 3
- 2024 NSC Diagnostics Book 1.pdf
- 2025 Diagnostic Report - Book 1.pdf
- Diagnostics Report 2023 Book 1.pdf


# Notebook 06 — Learner Error Mining & Diagnostic Difficulty

## Purpose
Extract official DBE NSC Mathematics diagnostic evidence on:
- average learner performance
- documented common errors
- documented misconceptions

## Evidence hierarchy
- Notebook 05 = **exposure** (what was assessed)
- Notebook 06 = **difficulty** (what learners struggled with)
- Notebook 07 = **priority** (exposure + difficulty + persistence)

## Hard rules
1. Keep DBE grain (do not invent subquestion splits)
2. Never invent performance percentages
3. Quote error text; summarise separately
4. Normalise misconception labels across years
5. Difficulty bands locked before looking at distributions

## Current diagnostic window
Start with verified 2023–2025 Book 1 Mathematics chapters.
Expand inventory toward 2014–2025 where comparable reports exist.

In [11]:
from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent

RAW_DIAG = PROJECT_ROOT / "data" / "raw" / "diagnostic_reports"
META_DIR = PROJECT_ROOT / "data" / "metadata"
PROC_DIAG = PROJECT_ROOT / "data" / "processed" / "diagnostics"
INTERIM_DIAG = PROJECT_ROOT / "data" / "interim" / "diagnostic_chapters"

for d in [RAW_DIAG, META_DIR, PROC_DIAG, INTERIM_DIAG]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw diagnostics:", RAW_DIAG)
print("PDFs currently present:")
pdfs = sorted(RAW_DIAG.glob("*.pdf"))
if not pdfs:
    print("  (none yet)")
else:
    for p in pdfs:
        print(f"  - {p.name} ({p.stat().st_size / 1e6:.2f} MB)")

Project root: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
Raw diagnostics: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\diagnostic_reports
PDFs currently present:
  - 2024 NSC Diagnostics Book 1.pdf (8.05 MB)
  - 2025 Diagnostic Report - Book 1.pdf (8.92 MB)
  - Diagnostic Report 2023 Book 1.......pdf (6.79 MB)


In [12]:
from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent

RAW_DIAG = PROJECT_ROOT / "data" / "raw" / "diagnostic_reports"
META_DIR = PROJECT_ROOT / "data" / "metadata"
PROC_DIAG = PROJECT_ROOT / "data" / "processed" / "diagnostics"
INTERIM_DIAG = PROJECT_ROOT / "data" / "interim" / "diagnostic_chapters"

for d in [RAW_DIAG, META_DIR, PROC_DIAG, INTERIM_DIAG]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw diagnostics:", RAW_DIAG)
print("PDFs currently present:")
pdfs = sorted(RAW_DIAG.glob("*.pdf"))
if not pdfs:
    print("  (none yet)")
else:
    for p in pdfs:
        print(f"  - {p.name} ({p.stat().st_size / 1e6:.2f} MB)")

Project root: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
Raw diagnostics: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\diagnostic_reports
PDFs currently present:
  - 2024_maths_diagnostic.pdf (8.05 MB)
  - 2025_maths_diagnostic.pdf (8.92 MB)
  - Diagnostic Report 2023 Book 1.......pdf (6.79 MB)


In [14]:
sources = [
    {
        "year": 2023,
        "filename": "2023_maths_diagnostic.pdf",
        "book": "Book 1",
        "chapter": "Chapter 10 — Mathematics",
        "expected_start_note": "~PDF page 211 (confirm manually)",
        "verified": False,
        "chapter_page_start": None,
        "chapter_page_end": None,
        "notes": "Must contain per-question performance + common errors",
    },
    {
        "year": 2024,
        "filename": "2024_maths_diagnostic.pdf",
        "book": "Book 1",
        "chapter": "Chapter 10 — Mathematics",
        "expected_start_note": "~PDF page 217 (confirm manually)",
        "verified": False,
        "chapter_page_start": None,
        "chapter_page_end": None,
        "notes": "Must contain per-question performance + common errors",
    },
    {
        "year": 2025,
        "filename": "2025_maths_diagnostic.pdf",
        "book": "Book 1",
        "chapter": "Chapter 10 — Mathematics",
        "expected_start_note": "~PDF page 229 (confirm manually)",
        "verified": False,
        "chapter_page_start": None,
        "chapter_page_end": None,
        "notes": "Must contain per-question performance + common errors",
    },
]

manifest_rows = []
for s in sources:
    path = RAW_DIAG / s["filename"]
    exists = path.exists()
    size_mb = round(path.stat().st_size / 1e6, 2) if exists else None
    manifest_rows.append({
        "year": s["year"],
        "filename": s["filename"],
        "path": str(path),
        "exists": exists,
        "size_mb": size_mb,
        "book": s["book"],
        "chapter": s["chapter"],
        "expected_start_note": s["expected_start_note"],
        "verified": s["verified"],
        "chapter_page_start": s["chapter_page_start"],
        "chapter_page_end": s["chapter_page_end"],
        "notes": s["notes"],
        "manifest_updated_at": datetime.now(timezone.utc).isoformat(),
    })

manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df)

# Save CSV + JSON
manifest_csv = META_DIR / "diagnostic_document_register.csv"
manifest_json = PROC_DIAG / "diagnostic_manifest.json"
manifest_df.to_csv(manifest_csv, index=False)
manifest_json.write_text(manifest_df.to_json(orient="records", indent=2), encoding="utf-8")

print("\nSaved:")
print(" -", manifest_csv)
print(" -", manifest_json)
print("\nGATE: Do not extract until verified=True for each available year.")
print("Open each PDF, find Mathematics chapter, fill page start/end, then update verified.")

,year,filename,path,exists,size_mb,book,chapter,expected_start_note,verified,chapter_page_start,chapter_page_end,notes,manifest_updated_at
0,2023,2023_maths_diagnostic.pdf,c:\Users\Administrator\Desktop\Matric-Maths-Ex...,True,6.79,Book 1,Chapter 10 — Mathematics,~PDF page 211 (confirm manually),False,None,None,Must contain per-question performance + common...,2026-09-12T11:58:26.598356+00:00
1,2024,2024_maths_diagnostic.pdf,c:\Users\Administrator\Desktop\Matric-Maths-Ex...,True,8.05,Book 1,Chapter 10 — Mathematics,~PDF page 217 (confirm manually),False,None,None,Must contain per-question performance + common...,2026-09-12T11:58:26.598792+00:00
2,2025,2025_maths_diagnostic.pdf,c:\Users\Administrator\Desktop\Matric-Maths-Ex...,True,8.92,Book 1,Chapter 10 — Mathematics,~PDF page 229 (confirm manually),False,None,None,Must contain per-question performance + common...,2026-09-12T11:58:26.599161+00:00



Saved:
 - c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\diagnostic_document_register.csv
 - c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\diagnostics\diagnostic_manifest.json

GATE: Do not extract until verified=True for each available year.
Open each PDF, find Mathematics chapter, fill page start/end, then update verified.


In [15]:
# Update verified page ranges
updates = {
    2023: {"start": 221, "end": 234},
    2024: {"start": 217, "end": 243},
    2025: {"start": 227, "end": 251},
}

manifest_df = pd.read_csv(META_DIR / "diagnostic_document_register.csv")

# ensure 2023 exists after rename
manifest_df["exists"] = manifest_df["filename"].apply(lambda f: (RAW_DIAG / f).exists())
manifest_df["size_mb"] = manifest_df["filename"].apply(
    lambda f: round((RAW_DIAG / f).stat().st_size / 1e6, 2) if (RAW_DIAG / f).exists() else None
)

for i, row in manifest_df.iterrows():
    y = int(row["year"])
    if y in updates and row["exists"]:
        manifest_df.at[i, "chapter_page_start"] = updates[y]["start"]
        manifest_df.at[i, "chapter_page_end"] = updates[y]["end"]
        manifest_df.at[i, "verified"] = True
        manifest_df.at[i, "notes"] = "Mathematics chapter page range verified manually"

manifest_df["manifest_updated_at"] = datetime.now(timezone.utc).isoformat()
display(manifest_df)

manifest_df.to_csv(META_DIR / "diagnostic_document_register.csv", index=False)
(PROC_DIAG / "diagnostic_manifest.json").write_text(
    manifest_df.to_json(orient="records", indent=2),
    encoding="utf-8"
)

print("Verified years:", manifest_df.loc[manifest_df["verified"] == True, "year"].tolist())
assert manifest_df["verified"].all(), "Not all sources verified"
print("GATE PASSED — ready for chapter extraction")

,year,filename,path,exists,size_mb,book,chapter,expected_start_note,verified,chapter_page_start,chapter_page_end,notes,manifest_updated_at
0,2023,2023_maths_diagnostic.pdf,c:\Users\Administrator\Desktop\Matric-Maths-Ex...,True,6.79,Book 1,Chapter 10 — Mathematics,~PDF page 211 (confirm manually),True,221.0,234.0,Mathematics chapter page range verified manually,2026-09-12T12:04:15.560165+00:00
1,2024,2024_maths_diagnostic.pdf,c:\Users\Administrator\Desktop\Matric-Maths-Ex...,True,8.05,Book 1,Chapter 10 — Mathematics,~PDF page 217 (confirm manually),True,217.0,243.0,Mathematics chapter page range verified manually,2026-09-12T12:04:15.560165+00:00
2,2025,2025_maths_diagnostic.pdf,c:\Users\Administrator\Desktop\Matric-Maths-Ex...,True,8.92,Book 1,Chapter 10 — Mathematics,~PDF page 229 (confirm manually),True,227.0,251.0,Mathematics chapter page range verified manually,2026-09-12T12:04:15.560165+00:00


Verified years: [2023, 2024, 2025]
GATE PASSED — ready for chapter extraction


In [16]:
try:
    import pdfplumber
except ImportError:
    import sys
    !{sys.executable} -m pip install pdfplumber
    import pdfplumber

def extract_chapter(pdf_path, start_page, end_page):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        last = min(end_page, len(pdf.pages))
        for i in range(start_page - 1, last):
            text = pdf.pages[i].extract_text() or ""
            pages.append({
                "pdf_page": i + 1,
                "char_count": len(text),
                "text": text,
            })
    return pages

extracted_summary = []

for _, row in manifest_df.iterrows():
    if not bool(row["verified"]):
        print(f"SKIP {row['year']}: not verified")
        continue

    pdf_path = Path(row["path"])
    start_p = int(row["chapter_page_start"])
    end_p = int(row["chapter_page_end"])

    print(f"Extracting {row['year']} pages {start_p}-{end_p} ...")
    pages = extract_chapter(pdf_path, start_p, end_p)
    df_pages = pd.DataFrame(pages)
    df_pages["year"] = int(row["year"])
    df_pages["source_file"] = row["filename"]

    out = INTERIM_DIAG / f"{int(row['year'])}_maths_chapter10.csv"
    df_pages.to_csv(out, index=False)

    extracted_summary.append({
        "year": int(row["year"]),
        "pages": len(df_pages),
        "total_chars": int(df_pages["char_count"].sum()),
        "output": str(out),
    })
    print(f"  saved {out.name} | pages={len(df_pages)} | chars={df_pages['char_count'].sum()}")

summary_df = pd.DataFrame(extracted_summary)
display(summary_df)

Extracting 2023 pages 221-234 ...
  saved 2023_maths_chapter10.csv | pages=14 | chars=33574
Extracting 2024 pages 217-243 ...
  saved 2024_maths_chapter10.csv | pages=27 | chars=63713
Extracting 2025 pages 227-251 ...
  saved 2025_maths_chapter10.csv | pages=25 | chars=60541


,year,pages,total_chars,output
0,2023,14,33574,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
1,2024,27,63713,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
2,2025,25,60541,c:\Users\Administrator\Desktop\Matric-Maths-Ex...


In [17]:
for _, row in summary_df.iterrows():
    sample = pd.read_csv(row["output"])
    text0 = str(sample.iloc[0]["text"])[:500]
    print("=" * 60)
    print(f"YEAR {row['year']} first page sample:")
    print(text0)
    print()

YEAR 2023 first page sample:
Mathematics
(d) In Q6.2.2 many candidates used the present value formula rather than the future value
formula.
(e) Most candidates in Q6.3 did not use logarithms correctly, and if they did, the
candidates rounded their answer of n = 147,8 to n = 148. This indicated a
misconception of the number of withdrawals of R20 000 that could have been made.
Suggestions for improvement
(a) Learners need deeper insight into the relevance of each of the formulae and under
which circumstances each formula can 

YEAR 2024 first page sample:
Mathematical Literacy
(e) Questions involving measurement and costing need to be practised.
(f) Class activities should be context based so that learners can develop the skill of solving
problems when given scenarios or visual texts.
(g) Districts need to develop question banks (on each topic) which can be used by
teachers after teaching the topics.
QUESTION 5: MAPS AND PLANS AND MEASUREMENT
Common errors and misconceptions
(a) Q5.1.1 w

In [18]:
import pdfplumber
from pathlib import Path

pdf_path = RAW_DIAG / "2024_maths_diagnostic.pdf"

hits = []
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        low = text.lower()
        if (
            "chapter 10" in low
            or ("mathematics" in low and "mathematical literacy" not in low[:200])
            or "differential calculus" in low
            or "euclidean geometry" in low
        ):
            hits.append({
                "pdf_page": i + 1,
                "preview": text[:180].replace("\n", " ")
            })

hits_df = pd.DataFrame(hits)
display(hits_df.head(30))
print("Total candidate pages:", len(hits_df))

,pdf_page,preview
0,2,CONTENTS – PART 1 Foreword by the Minister 1 C...
1,3,"Diagnostic Report 2024: Foreword, Introduction..."
2,12,"Diagnostic Report 2024: Foreword, Introduction..."
3,24,Accounting learners to appreciate different ap...
4,126,Geography (w) The responses to questions in Se...
5,219,Mathematics CHAPTER 10 MATHEMATICS The followi...
6,220,Mathematics Graph 10.1.1 Overall achievement r...
7,221,Mathematics 10.2 OVERVIEW OF CANDIDATES' PERFO...
8,222,Mathematics Graph 10.3.2 Average performance p...
9,223,Mathematics standard form of 2𝑥2+1−4𝑥 = 0 and ...


Total candidate pages: 32


In [19]:
updates = {
    2023: {"start": 213, "end": 236},
    2024: {"start": 219, "end": 245},
    2025: {"start": 230, "end": 254},
}

manifest_df = pd.read_csv(META_DIR / "diagnostic_document_register.csv")
manifest_df["exists"] = manifest_df["filename"].apply(lambda f: (RAW_DIAG / f).exists())
manifest_df["size_mb"] = manifest_df["filename"].apply(
    lambda f: round((RAW_DIAG / f).stat().st_size / 1e6, 2) if (RAW_DIAG / f).exists() else None
)

for i, row in manifest_df.iterrows():
    y = int(row["year"])
    if y in updates and bool(row["exists"]):
        manifest_df.at[i, "chapter_page_start"] = updates[y]["start"]
        manifest_df.at[i, "chapter_page_end"] = updates[y]["end"]
        manifest_df.at[i, "verified"] = True
        manifest_df.at[i, "notes"] = "Mathematics chapter verified by content search"

manifest_df["manifest_updated_at"] = datetime.now(timezone.utc).isoformat()
display(manifest_df[["year", "filename", "exists", "size_mb", "verified", "chapter_page_start", "chapter_page_end"]])

manifest_df.to_csv(META_DIR / "diagnostic_document_register.csv", index=False)
(PROC_DIAG / "diagnostic_manifest.json").write_text(
    manifest_df.to_json(orient="records", indent=2), encoding="utf-8"
)
print("Verified years:", manifest_df.loc[manifest_df["verified"] == True, "year"].tolist())

,year,filename,exists,size_mb,verified,chapter_page_start,chapter_page_end
0,2023,2023_maths_diagnostic.pdf,True,6.79,True,213.0,236.0
1,2024,2024_maths_diagnostic.pdf,True,8.05,True,219.0,245.0
2,2025,2025_maths_diagnostic.pdf,True,8.92,True,230.0,254.0


Verified years: [2023, 2024, 2025]


In [20]:
try:
    import pdfplumber
except ImportError:
    import sys
    !{sys.executable} -m pip install pdfplumber
    import pdfplumber

def extract_chapter(pdf_path, start_page, end_page):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        last = min(end_page, len(pdf.pages))
        for i in range(start_page - 1, last):
            text = pdf.pages[i].extract_text() or ""
            pages.append({
                "pdf_page": i + 1,
                "char_count": len(text),
                "text": text,
            })
    return pages

extracted_summary = []

for _, row in manifest_df.iterrows():
    if not bool(row["verified"]):
        print(f"SKIP {row['year']}: not verified")
        continue

    pdf_path = Path(row["path"])
    start_p = int(row["chapter_page_start"])
    end_p = int(row["chapter_page_end"])

    print(f"Extracting {row['year']} pages {start_p}-{end_p} ...")
    pages = extract_chapter(pdf_path, start_p, end_p)
    df_pages = pd.DataFrame(pages)
    df_pages["year"] = int(row["year"])
    df_pages["source_file"] = row["filename"]

    out = INTERIM_DIAG / f"{int(row['year'])}_maths_chapter10.csv"
    df_pages.to_csv(out, index=False)

    extracted_summary.append({
        "year": int(row["year"]),
        "pages": len(df_pages),
        "total_chars": int(df_pages["char_count"].sum()),
        "output": str(out),
    })
    print(f"  saved {out.name} | pages={len(df_pages)} | chars={df_pages['char_count'].sum()}")

summary_df = pd.DataFrame(extracted_summary)
display(summary_df)

Extracting 2023 pages 213-236 ...
  saved 2023_maths_chapter10.csv | pages=24 | chars=53364
Extracting 2024 pages 219-245 ...
  saved 2024_maths_chapter10.csv | pages=27 | chars=65156
Extracting 2025 pages 230-254 ...
  saved 2025_maths_chapter10.csv | pages=25 | chars=60278


,year,pages,total_chars,output
0,2023,24,53364,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
1,2024,27,65156,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
2,2025,25,60278,c:\Users\Administrator\Desktop\Matric-Maths-Ex...


In [21]:
for _, row in summary_df.iterrows():
    sample = pd.read_csv(row["output"])
    text0 = str(sample.iloc[0]["text"])[:500]
    print("=" * 60)
    print(f"YEAR {row['year']} first page sample:")
    print(text0)
    print()

YEAR 2023 first page sample:
Mathematics
CHAPTER 10
MATHEMATICS
The following report should be read in conjunction with the Mathematics question
papers for the NSC November 2023 examinations.
10.1 PERFORMANCE TRENDS (2019–2023)
The number of candidates who wrote the Mathematics examination in 2023 decreased by
7 718 compared to that of 2022.
There was a significant improvement in the pass rate this year. Candidates who passed at the
30% level improved from 55% in 2022 to 63,5% in 2023. There was a corresponding
improvement 

YEAR 2024 first page sample:
Mathematics
CHAPTER 10
MATHEMATICS
The following report should be read in conjunction with the Mathematics question
papers for the NSC November 2024 examinations.
10.1 PERFORMANCE TRENDS (2020–2024)
The number of candidates who sat for the Mathematics examinations in 2024 decreased by
10 528, compared to that of 2023.
There was a significant improvement in the pass rate this year. Candidates who passed at the
30% level improved from 63,

In [22]:
sample = pd.read_csv(INTERIM_DIAG / "2024_maths_chapter10.csv")
print(sample.iloc[0]["text"][:600])

Mathematics
CHAPTER 10
MATHEMATICS
The following report should be read in conjunction with the Mathematics question
papers for the NSC November 2024 examinations.
10.1 PERFORMANCE TRENDS (2020–2024)
The number of candidates who sat for the Mathematics examinations in 2024 decreased by
10 528, compared to that of 2023.
There was a significant improvement in the pass rate this year. Candidates who passed at the
30% level improved from 63,5% in 2023 to 69,1% in 2024. There was a corresponding
improvement in the pass rate at the 40% level over the past two years from 43,6% to 47,9%.
The percentage


In [23]:
import re
import pandas as pd
from pathlib import Path

chapter_files = sorted(INTERIM_DIAG.glob("*_maths_chapter10.csv"))
print("Chapter files:", [p.name for p in chapter_files])

chapters = []
for f in chapter_files:
    df = pd.read_csv(f)
    year = int(str(f.name)[:4])
    full_text = "\n\n".join(df["text"].fillna("").astype(str).tolist())
    chapters.append({
        "year": year,
        "n_pages": len(df),
        "full_text": full_text,
        "source_file": f.name,
    })

chapters_df = pd.DataFrame(chapters)
display(chapters_df[["year", "n_pages", "source_file"]])

Chapter files: ['2023_maths_chapter10.csv', '2024_maths_chapter10.csv', '2025_maths_chapter10.csv']


,year,n_pages,source_file
0,2023,24,2023_maths_chapter10.csv
1,2024,27,2024_maths_chapter10.csv
2,2025,25,2025_maths_chapter10.csv


In [24]:
def split_question_blocks(text: str, year: int):
    """
    Split Mathematics diagnostic chapter into QUESTION blocks.
    Keeps DBE grain as much as possible.
    """
    # Normalise
    t = text.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)

    # Split on QUESTION headings
    pattern = re.compile(
        r"(?=QUESTION\s+\d+\s*[:\-])",
        flags=re.IGNORECASE
    )
    parts = pattern.split(t)

    rows = []
    for part in parts:
        part = part.strip()
        if not part:
            continue

        m = re.match(
            r"QUESTION\s+(\d+)\s*[:\-]?\s*(.*)",
            part,
            flags=re.IGNORECASE | re.DOTALL
        )
        if not m:
            # front matter / overview before first QUESTION
            rows.append({
                "year": year,
                "question_number": None,
                "question_title": "OVERVIEW",
                "block_text": part[:5000],
            })
            continue

        qn = int(m.group(1))
        rest = m.group(2).strip()
        title_line = rest.split("\n", 1)[0][:120].strip()

        rows.append({
            "year": year,
            "question_number": qn,
            "question_title": title_line,
            "block_text": part[:12000],
        })
    return rows

block_rows = []
for _, row in chapters_df.iterrows():
    block_rows.extend(split_question_blocks(row["full_text"], int(row["year"])))

blocks_df = pd.DataFrame(block_rows)
print("Blocks:", len(blocks_df))
display(blocks_df.head(20))

Blocks: 68


,year,question_number,question_title,block_text
0,2023,NaN,OVERVIEW,Mathematics\nCHAPTER 10\nMATHEMATICS\nThe foll...
1,2023,1.0,ALGEBRA,QUESTION 1: ALGEBRA\nCommon errors and misconc...
2,2023,2.0,PATTERNS,QUESTION 2: PATTERNS\nCommon errors and miscon...
3,2023,3.0,PATTERNS,QUESTION 3: PATTERNS\nCommon errors and miscon...
4,2023,4.0,FUNCTIONS (EXPONENTIAL AND LOGARITHMIC GRAPH),QUESTION 4: FUNCTIONS (EXPONENTIAL AND LOGARIT...
5,2023,5.0,FUNCTIONS (HYPERBOLA AND PARABOLA),QUESTION 5: FUNCTIONS (HYPERBOLA AND PARABOLA)...
6,2023,6.0,FINANCE,QUESTION 6: FINANCE\nCommon errors and misconc...
7,2023,7.0,CALCULUS,QUESTION 7: CALCULUS\nCommon errors and miscon...
8,2023,8.0,CALCULUS,QUESTION 8: CALCULUS\nCommon errors and miscon...
9,2023,9.0,CALCULUS,QUESTION 9: CALCULUS\nCommon error and misconc...


In [25]:
def extract_performance(block_text: str):
    """
    Pull explicit percentage mentions near performance language.
    Leave null if not explicit.
    """
    if not isinstance(block_text, str):
        return None, None

    candidates = []

    # Patterns like: average of 34%, average performance was 69%, obtained 41%
    patterns = [
        r"average(?:\s+performance)?(?:\s+was|\s+of)?\s+(\d{1,3})\s*%",
        r"obtained\s+an\s+average\s+of\s+(\d{1,3})\s*%",
        r"average\s+was\s+(\d{1,3})\s*%",
        r"performance\s+was\s+(\d{1,3})\s*%",
        r"\((\d{1,3})%\)",
    ]

    for pat in patterns:
        for m in re.finditer(pat, block_text, flags=re.IGNORECASE):
            val = int(m.group(1))
            if 0 <= val <= 100:
                candidates.append(val)

    if not candidates:
        return None, None

    # Keep unique values; if many, store all as list-string
    uniq = sorted(set(candidates))
    primary = uniq[0] if len(uniq) == 1 else None
    return primary, ";".join(map(str, uniq))


perf_rows = []
for _, r in blocks_df.iterrows():
    primary, all_vals = extract_performance(r["block_text"])
    perf_rows.append({
        "year": r["year"],
        "question_number": r["question_number"],
        "question_title": r["question_title"],
        "avg_performance_pct": primary,
        "all_pct_mentions": all_vals,
        "grain": "question" if pd.notna(r["question_number"]) else "overview",
    })

performance_df = pd.DataFrame(perf_rows)
print("Performance rows:", len(performance_df))
print("With numeric avg:", performance_df["avg_performance_pct"].notna().sum())
display(performance_df.head(25))

Performance rows: 68
With numeric avg: 0


,year,question_number,question_title,avg_performance_pct,all_pct_mentions,grain
0,2023,NaN,OVERVIEW,None,None,overview
1,2023,1.0,ALGEBRA,None,None,question
2,2023,2.0,PATTERNS,None,None,question
3,2023,3.0,PATTERNS,None,None,question
4,2023,4.0,FUNCTIONS (EXPONENTIAL AND LOGARITHMIC GRAPH),None,None,question
5,2023,5.0,FUNCTIONS (HYPERBOLA AND PARABOLA),None,None,question
6,2023,6.0,FINANCE,None,None,question
7,2023,7.0,CALCULUS,None,None,question
8,2023,8.0,CALCULUS,None,None,question
9,2023,9.0,CALCULUS,None,None,question


In [26]:
def extract_error_section(block_text: str):
    if not isinstance(block_text, str):
        return "", ""

    # Capture from "Common errors..." until Suggestions / next QUESTION / end
    m = re.search(
        r"(Common errors and misconceptions)(.*?)(?:Suggestions for improvement|QUESTION\s+\d+|\Z)",
        block_text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if not m:
        return "", ""

    section = m.group(2).strip()
    return section, section[:1500]


def split_error_bullets(section: str):
    if not section:
        return []
    # Split on (a) (b) (c) style bullets
    parts = re.split(r"\n?\s*\(([a-z])\)\s+", section, flags=re.IGNORECASE)
    # parts alternate: preamble, letter, text, letter, text...
    bullets = []
    if len(parts) == 1:
        cleaned = " ".join(section.split())
        if len(cleaned) > 20:
            bullets.append(cleaned)
        return bullets

    # first chunk may be preamble
    i = 1
    while i + 1 < len(parts):
        letter = parts[i]
        text = " ".join(parts[i + 1].split())
        if len(text) > 15:
            bullets.append(f"({letter}) {text}")
        i += 2
    return bullets


error_rows = []
for _, r in blocks_df.iterrows():
    section, _ = extract_error_section(r["block_text"])
    bullets = split_error_bullets(section)
    if not bullets and section:
        bullets = [" ".join(section.split())[:1000]]

    for b in bullets:
        error_rows.append({
            "year": r["year"],
            "question_number": r["question_number"],
            "question_title": r["question_title"],
            "error_text_raw": b,
            "error_text_summary": None,  # fill later if needed
            "source": "DBE diagnostic",
        })

errors_df = pd.DataFrame(error_rows)
print("Error rows:", len(errors_df))
display(errors_df.head(20))

Error rows: 273


,year,question_number,question_title,error_text_raw,error_text_summary,source
0,2023,1.0,ALGEBRA,(a) In Q1.1.2 some candidates did not write th...,None,DBE diagnostic
1,2023,1.0,ALGEBRA,(b) In Q1.1.3 many candidates were able to squ...,None,DBE diagnostic
2,2023,1.0,ALGEBRA,(c) Many candidates struggled to solve the ine...,None,DBE diagnostic
3,2023,1.0,ALGEBRA,(d) The simultaneous equation given in fractio...,None,DBE diagnostic
4,2023,1.0,ALGEBRA,(e) In Q1.3 many candidates were able to write...,None,DBE diagnostic
5,2023,2.0,PATTERNS,(a) The most common errors in Q2.1 and its sub...,None,DBE diagnostic
6,2023,2.0,PATTERNS,(b) Many candidates substituted n = 5 into the...,None,DBE diagnostic
7,2023,2.0,PATTERNS,(c) In Q2.2.3 most candidates could not show t...,None,DBE diagnostic
8,2023,3.0,PATTERNS,(a) Candidates were correctly able to determin...,None,DBE diagnostic
9,2023,3.0,PATTERNS,(b) Many candidates did not make the link betw...,None,DBE diagnostic


In [27]:
# Locked before tuning
def difficulty_band(pct):
    if pd.isna(pct):
        return "unknown"
    if pct < 30:
        return "severe"
    if pct < 50:
        return "weak"
    if pct < 70:
        return "moderate"
    return "strong"

performance_df["difficulty_band"] = performance_df["avg_performance_pct"].apply(difficulty_band)
display(performance_df["difficulty_band"].value_counts(dropna=False))

difficulty_band
unknown    68
Name: count, dtype: int64

In [28]:
OUT = PROC_DIAG
OUT.mkdir(parents=True, exist_ok=True)

blocks_df.to_csv(OUT / "diagnostic_question_blocks_v1.csv", index=False)
performance_df.to_csv(OUT / "diagnostic_performance_v1.csv", index=False)
errors_df.to_csv(OUT / "diagnostic_errors_v1.csv", index=False)

print("Saved:")
print(" - diagnostic_question_blocks_v1.csv", len(blocks_df))
print(" - diagnostic_performance_v1.csv", len(performance_df))
print(" - diagnostic_errors_v1.csv", len(errors_df))

Saved:
 - diagnostic_question_blocks_v1.csv 68
 - diagnostic_performance_v1.csv 68
 - diagnostic_errors_v1.csv 273


In [29]:
def assign_paper(groups):
    rows = []
    for year, g in groups:
        g = g.sort_index().copy()
        # First OVERVIEW + first run of questions = P1; after Probability/Data shift often P2
        paper = "P1"
        seen_prob = False
        out = []
        for _, r in g.iterrows():
            title = str(r["question_title"]).upper()
            qn = r["question_number"]

            # Heuristic: once DATA HANDLING / ANALYTICAL / EUCLIDEAN appears after P1 topics, switch to P2
            if any(k in title for k in ["DATA HANDLING", "ANALYTICAL GEOMETRY", "EUCLIDEAN", "TRIGONOMETRY"]) and paper == "P1":
                # only switch if we already passed some P1 content
                if len(out) >= 5:
                    paper = "P2"

            if "PROBABILITY" in title:
                seen_prob = True

            rr = r.to_dict()
            rr["paper"] = None if pd.isna(qn) and str(r["question_title"]).upper() == "OVERVIEW" else paper
            out.append(rr)
        rows.extend(out)
    return pd.DataFrame(rows)

blocks_df = assign_paper(blocks_df.groupby("year"))
display(blocks_df.groupby(["year", "paper"], dropna=False).size())

year  paper
2023  P1       10
      P2       10
      NaN       1
2024  P1       12
      P2       11
      NaN       1
2025  P1       11
      P2       11
      NaN       1
dtype: int64

In [30]:
def extract_pct_mentions(text):
    if not isinstance(text, str):
        return []
    pats = [
        r"average(?:\s+performance)?(?:\s+was|\s+of)?\s+(\d{1,3})\s*%",
        r"obtained\s+(?:an\s+)?average\s+of\s+(\d{1,3})\s*%",
        r"performance\s+was\s+(\d{1,3})\s*%",
        r"average\s+of\s+(\d{1,3})\s*%",
        r"(\d{1,3})\s*%\s+(?:in|for|on)\s+Q\s*\d+",
        r"Q\s*\d+[.\d]*[^%]{0,40}?(\d{1,3})\s*%",
    ]
    vals = []
    for pat in pats:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            v = int(m.group(1))
            if 0 <= v <= 100:
                vals.append(v)
    return sorted(set(vals))

# Search full chapter texts
chapter_pct = []
for _, row in chapters_df.iterrows():
    vals = extract_pct_mentions(row["full_text"])
    chapter_pct.append({
        "year": row["year"],
        "n_pct_values_found": len(vals),
        "pct_values": vals[:30],
    })
display(pd.DataFrame(chapter_pct))

,year,n_pct_values_found,pct_values
0,2023,0,[]
1,2024,0,[]
2,2025,0,[]


In [31]:
SEVERE_TERMS = [
    "unable", "could not", "did not understand", "poorly answered",
    "majority of candidates did not", "most candidates were unable",
    "not understood", "struggled", "failed to",
]
MODERATE_TERMS = [
    "some candidates", "many candidates", "common error",
    "incorrectly", "forgot", "left their answer",
]

def qualitative_severity(text):
    t = (text or "").lower()
    severe_hits = sum(1 for w in SEVERE_TERMS if w in t)
    moderate_hits = sum(1 for w in MODERATE_TERMS if w in t)
    if severe_hits >= 1:
        return "severe_language"
    if moderate_hits >= 1:
        return "moderate_language"
    return "mentioned"

errors_df["qualitative_severity"] = errors_df["error_text_raw"].apply(qualitative_severity)
display(errors_df["qualitative_severity"].value_counts())

qualitative_severity
moderate_language    134
severe_language      119
mentioned             20
Name: count, dtype: int64

In [32]:
TOPIC_MAP = {
    "ALGEBRA": "Algebra & Equations",
    "PATTERNS": "Number Patterns & Sequences",
    "FUNCTIONS": "Functions & Graphs",
    "FINANCE": "Finance",
    "CALCULUS": "Calculus",
    "PROBABILITY": "Probability",
    "DATA HANDLING": "Statistics",
    "STATISTICS": "Statistics",
    "ANALYTICAL GEOMETRY": "Analytical Geometry",
    "TRIGONOMETRY": "Trigonometry",
    "EUCLIDEAN": "Euclidean Geometry",
}

def map_title_to_topic(title):
    t = str(title).upper()
    for k, v in TOPIC_MAP.items():
        if k in t:
            return v
    return "Unmapped"

errors_df["topic"] = errors_df["question_title"].apply(map_title_to_topic)
performance_df["topic"] = performance_df["question_title"].apply(map_title_to_topic)

print(errors_df["topic"].value_counts())

topic
Trigonometry                   50
Analytical Geometry            40
Functions & Graphs             39
Euclidean Geometry             32
Statistics                     31
Number Patterns & Sequences    20
Calculus                       17
Algebra & Equations            15
Finance                        13
Probability                    13
Unmapped                        3
Name: count, dtype: int64


In [33]:
error_pressure = (
    errors_df.groupby("topic")
    .agg(
        error_count=("error_text_raw", "count"),
        severe_count=("qualitative_severity", lambda s: (s == "severe_language").sum()),
        years=("year", "nunique"),
    )
    .reset_index()
    .sort_values(["severe_count", "error_count"], ascending=False)
)

display(error_pressure)
error_pressure.to_csv(PROC_DIAG / "diagnostic_error_pressure_v1.csv", index=False)
errors_df.to_csv(PROC_DIAG / "diagnostic_errors_v1.csv", index=False)

,topic,error_count,severe_count,years
9,Trigonometry,50,27,3
1,Analytical Geometry,40,17,3
5,Functions & Graphs,39,16,3
8,Statistics,31,16,3
3,Euclidean Geometry,32,11,3
0,Algebra & Equations,15,8,3
7,Probability,13,7,3
6,Number Patterns & Sequences,20,6,3
4,Finance,13,5,3
2,Calculus,17,4,3


In [34]:
error_pressure.to_csv(PROC_DIAG / "diagnostic_error_pressure_v1.csv", index=False)
errors_df.to_csv(PROC_DIAG / "diagnostic_errors_v1.csv", index=False)
blocks_df.to_csv(PROC_DIAG / "diagnostic_question_blocks_v1.csv", index=False)

summary = f"""
# Notebook 06 Summary

Years: 2023–2025
Error records: {len(errors_df)}
Severe-language records: {(errors_df['qualitative_severity']=='severe_language').sum()}
Numeric averages recovered from text: 0 (graph-based in source PDFs)

## Highest diagnostic error pressure
{error_pressure.head(5).to_string(index=False)}

## Limitation
Difficulty v1 uses documented error commentary, not national mean percentages.
"""
(PROC_DIAG / "notebook06_summary.md").write_text(summary, encoding="utf-8")
print(summary)
print("NOTEBOOK 06 v1 COMPLETE")


# Notebook 06 Summary

Years: 2023–2025
Error records: 273
Severe-language records: 119
Numeric averages recovered from text: 0 (graph-based in source PDFs)

## Highest diagnostic error pressure
              topic  error_count  severe_count  years
       Trigonometry           50            27      3
Analytical Geometry           40            17      3
 Functions & Graphs           39            16      3
         Statistics           31            16      3
 Euclidean Geometry           32            11      3

## Limitation
Difficulty v1 uses documented error commentary, not national mean percentages.

NOTEBOOK 06 v1 COMPLETE
